# Case pipeline_lab — Churn (Databricks)

Funil completo, uma etapa por célula: `divisao → construcao → esfera1 → esfera2 → categorizar → preselecao → treinamento`.

Cada chamada mostra **todos os parâmetros**, mesmo os que ficam no valor default — assim serve de referência do que dá pra ajustar. Ver `python/pipeline_lab/REFERENCIA.md` (ou o PDF em `docs/pipeline_lab_referencia.pdf`) para a explicação de cada parâmetro e a literatura por trás.

## 0. Instalação

In [ ]:
%pip install --upgrade --force-reinstall git+https://github.com/PedroMaiorano/modelagem-lab.git

In [ ]:
dbutils.library.restartPython()

## 1. Carregar os dados (Spark → pandas)

`sample_submission`/`test` são a base sem rótulo (Kaggle-style) — usadas só no fim, para gerar a submissão. O treinamento roda inteiro sobre `train`, que tem `Churn`.

In [ ]:
sample_submission = spark.table("workspace.default.churn_sample_submission")
train = spark.table("workspace.default.churn_train")
test = spark.table("workspace.default.churn_test")

## 2. Preparar o alvo e o split dev/teste

`Churn` ("Yes"/"No") vira 0/1. O split usa uma única coluna de sorteio (`_sorteio`) reaproveitada nas duas comparações — chamar `F.rand(seed=42)` de novo em cada `.when()` sorteia números **diferentes** a cada chamada, mesmo com a mesma seed, e quebra a proporção 70/15/15 pretendida (o segundo corte vira independente do primeiro, não condicional).

In [ ]:
from pyspark.sql import functions as F

train = train.withColumn("Churn", F.when(F.col("Churn") == "Yes", 1).otherwise(0))

train = train.withColumn("_sorteio", F.rand(seed=42))
train = train.withColumn(
    "split",
    F.when(F.col("_sorteio") < 0.70, "train")
     .when(F.col("_sorteio") < 0.85, "val")
     .otherwise("test"),
).drop("_sorteio")

train_pd = train.toPandas()
test_pd = test.toPandas()

train_pd["split"].value_counts()

## 3. Divisão dev/teste (`pipeline_lab.divisao`)

`dividir_por_amostra` usa a coluna `split` que já veio do passo anterior. `coluna_y="Churn"` já renomeia o alvo para `"y"` — nome que todo o resto do funil exige. O bucket `"val"` fica de fora deste split (reservado como holdout extra — ver nota no fim do notebook, a lib ainda não expõe uma função de "aplicar transformação em dado novo sem re-ajustar" para um terceiro conjunto).

In [ ]:
from pipeline_lab import categorizar, construcao, divisao, esfera2, preselecao, treinamento

df_dev, df_teste = divisao.dividir_por_amostra(
    df=train_pd,
    coluna_amostra="split",
    valores_dev=["train"],
    valores_teste=["test"],
    coluna_y="Churn",
)

print(f"dev: {len(df_dev)} linhas | teste: {len(df_teste)} linhas")

## 3.5 Remover colunas de identificação (ID)

Uma coluna de ID (ex.: `customerID`) tem um valor **diferente por linha** — em `categorizar` ela cai no caminho "categórica" (cada valor vira seu próprio bin, sem discretização), e como os IDs de dev e teste não se repetem, praticamente TODO valor de teste vira "bin ausente da tabela de WOE" (WOE=0). É a causa mais comum do aviso `"N valor(es) com bin ausente da tabela de WOE"` aparecer com um número gigante — ver a nota completa lá na frente. Removendo aqui evita o ruído (a coluna não carregava sinal preditivo real de qualquer forma) e evita o desperdício de processar uma "categórica" com milhares de categorias únicas nas etapas seguintes.

In [ ]:
# Heurística: nome contém "id" OU todo valor é único (cardinalidade == nº de linhas).
# Confira a lista antes de rodar em produção -- heurística pode errar
# (ex.: uma coluna numérica contínua rara também pode ter cardinalidade == n).
colunas_id = [
    c
    for c in df_dev.columns
    if c != "y" and ("id" in c.lower() or df_dev[c].nunique(dropna=True) == len(df_dev))
]
print(f"Colunas removidas por parecerem ID: {colunas_id}")

df_dev = df_dev.drop(columns=colunas_id)
df_teste = df_teste.drop(columns=[c for c in colunas_id if c in df_teste.columns])

## 4. Construção de variáveis (`pipeline_lab.construcao`) — opcional

Duas formas de usar:

1. **`construir_razoes_em_lote`** — você lista os pares com sentido de negócio (ex.: `TotalCharges / tenure` ≡ "gasto médio mensal"), nome que você escolhe. Mais interpretável, exige conhecer o schema.
2. **`construir_todas_as_razoes`** — gera razão (e diferença) para **todo par ordenado** de colunas numéricas automaticamente, com nome genérico (`"{a}_sobre_{b}"`, `"{a}_menos_{b}"`). Não exige saber o schema, mas cresce quadraticamente (`n*(n-1)` novas colunas) — rode a pré-seleção (célula 8) logo depois pra filtrar o que não prestou.

Uso aqui: opção 2, já que não conheço o schema exato de `churn_train` neste ambiente.

In [ ]:
import pandas as pd

# Mesma lista de colunas nos dois lados -- garante que dev/teste ganhem
# exatamente as mesmas colunas novas, mesmo que algum dtype numérico
# divirja entre as duas bases por acaso.
colunas_numericas = [c for c in df_dev.columns if pd.api.types.is_numeric_dtype(df_dev[c])]

novas_dev = construcao.construir_todas_as_razoes(
    df=df_dev,
    colunas=colunas_numericas,
    incluir_diferenca=True,
    epsilon=1e-6,
)
novas_teste = construcao.construir_todas_as_razoes(
    df=df_teste,
    colunas=colunas_numericas,
    incluir_diferenca=True,
    epsilon=1e-6,
)
df_dev = pd.concat([df_dev, novas_dev], axis=1)
df_teste = pd.concat([df_teste, novas_teste], axis=1)

print(f"{novas_dev.shape[1]} razões/diferenças geradas automaticamente a partir de {len(colunas_numericas)} colunas numéricas")
df_dev.shape, df_teste.shape

## 5. Esfera 1 — agregação temporal (`pipeline_lab.esfera1`) — não aplicável aqui

Esfera 1 exige um **painel** (várias linhas por chave ao longo do tempo — ex.: atraso mês a mês do mesmo cliente). O dataset de churn aqui é uma linha por cliente (snapshot), não um painel, então esta etapa é pulada. Fica só como referência de assinatura, caso um dia você tenha um painel de uso mensal do cliente para agregar antes do resto do funil:

```python
from pipeline_lab import esfera1

df_dev, df_teste, colunas_geradas = esfera1.aplicar(
    df_dev=df_dev,
    df_teste=df_teste,
    chave="id_cliente",
    coluna_tempo="safra",
    colunas_valor=["uso_mensal"],
    janelas=[3, 6, 12],
)
```

## 6. Esfera 2 — descoberta de interação (`pipeline_lab.esfera2`)

RuleFit-style: descobre regras tipo "A > x E B > y" via um ensemble de árvores rasas, materializa como coluna 0/1.

In [ ]:
df_dev, df_teste, colunas_regra = esfera2.aplicar(
    df_dev=df_dev,
    df_teste=df_teste,
    colunas_categoricas=None,
    profundidade_maxima=2,
    n_arvores=60,
    min_suporte=0.02,
    max_suporte=0.5,
    max_regras=10,
    permitir_cruzamento_entre_bases=True,
    proporcao_variaveis_por_split=None,
    iv_minimo=0.02,
)

print(f"Regras de interação estáveis encontradas: {len(colunas_regra)}")
colunas_regra

## 7. Categorização + WOE (`pipeline_lab.categorizar`)

Sempre a última etapa antes da pré-seleção/treinamento — gera as versões `_woe`/`_log`/`_bin`/etc. de cada candidata.

**Sobre o aviso `"N valor(es) com bin ausente da tabela de WOE — atribuído WOE=0"`**: é o `logger.warning` de `transformacao.woe.aplicar_woe` (não um erro) — dispara quando a base de **teste** tem um valor/categoria que a tabela de WOE, ajustada só em **dev**, nunca viu. Comportamento esperado por design anti-leakage (WOE nunca é reajustado em teste); recebe WOE=0 (neutro) em vez de quebrar. Um número **pequeno** é normal (categoria rara que só apareceu em teste por acaso amostral). Um número **gigante** (dezenas de milhares) quase sempre significa uma coluna de **alta cardinalidade** (ID, chave única, texto livre) sendo tratada como categórica — cada valor de teste é "novo" porque nunca se repete. É pra isso que a célula 3.5 acima remove colunas de ID antes de chegar aqui.

In [ ]:
def _log_progresso(coluna: str, iv: float) -> None:
    print(f"  {coluna}: IV={iv:.4f}")


woe_dev, woe_teste, iv_por_variavel = categorizar.categorizar_e_transformar(
    df_dev=df_dev,
    df_teste=df_teste,
    gerar_transformacoes_potencia=True,
    gerar_bin_ordinal=True,
    ao_processar_coluna=_log_progresso,
)

print(f"woe_dev: {woe_dev.shape} | woe_teste: {woe_teste.shape}")

# Display estruturado: variável, IV, e a régua de interpretação de Siddiqi
# (classificar_iv) -- dá pra rodar display() no Databricks direto nisto.
resumo_iv = (
    pd.Series(iv_por_variavel, name="iv")
    .sort_values(ascending=False)
    .to_frame()
    .assign(classificacao=lambda d: d["iv"].map(categorizar.classificar_iv))
)
display(resumo_iv)

## 8. Pré-seleção (`pipeline_lab.preselecao`)

Filtra variância → IV → correlação, nesta ordem, antes de entregar ao Pedro_Wise.

In [ ]:
resultado_selecao = preselecao.pre_selecionar(
    df=woe_dev,
    iv_por_base=iv_por_variavel,
    limiar_variancia=1e-6,
    limiar_iv=0.02,
    limiar_correlacao=0.9,
)

print(
    f"candidatas: {resultado_selecao['n_inicial']} -> "
    f"{resultado_selecao['n_apos_variancia']} (variância) -> "
    f"{resultado_selecao['n_apos_iv']} (IV) -> "
    f"{resultado_selecao['n_final']} (correlação)"
)

print(f"\nDescartadas por variância quase-nula ({len(resultado_selecao['colunas_descartadas_variancia'])}):")
print(f"  {resultado_selecao['colunas_descartadas_variancia']}")

print(f"\nDescartadas por IV baixo ({len(resultado_selecao['colunas_descartadas_iv'])}):")
print(f"  {resultado_selecao['colunas_descartadas_iv']}")

print(f"\nDescartadas por correlação alta com uma candidata melhor ({len(resultado_selecao['pares_correlacionados_descartados'])}):")
display(pd.DataFrame(resultado_selecao["pares_correlacionados_descartados"], columns=["mantida", "descartada", "correlacao"]))

print(f"\nMantidas ({resultado_selecao['n_final']}):")
print(f"  {resultado_selecao['colunas_mantidas']}")

woe_dev = woe_dev[[*resultado_selecao["colunas_mantidas"], "y"]]
woe_teste = woe_teste[[*resultado_selecao["colunas_mantidas"], "y"]]

## 9. Treinamento — Pedro_Wise (`pipeline_lab.treinamento`)

A busca stepwise de verdade (forward/backward, níveis 1 a 3). `criterio="teste"` é o default recomendado (evita o viés de aceitar ruído que `criterio="dev"` tem — ver `REFERENCIA.md`).

`verbose=True` (default) imprime cada passo aceito **ao vivo**, enquanto a busca roda — qual etapa (`forward_simples`, `transformacao_simples`, `backward_simples`, nível 1/2/2.5/3), qual variável entrou/saiu, e o KS resultante naquele passo. `resultado.historico` guarda a mesma lista de eventos pra revisar depois de terminado.

In [ ]:
resultado = treinamento.treinar(
    df_dev=woe_dev,
    df_teste=woe_teste,
    criterio="teste",
    shadow_probing=False,
    forward_simples=True,
    transformacao_simples_nivel1=True,
    backward_simples_nivel1=True,
    min_vars_para_backward=5,
    forward_duplo=True,
    forward_triplo=True,
    transformacao_simples_nivel2=True,
    backward_simples_nivel2=True,
    n_best_duplo=5,
    n_best_triplo_1=3,
    n_best_triplo_2=3,
    nivel3_ativado=False,
    n_best_backward=2,
    profundidade_maxima_nivel3=2,
    p_valor_maximo=None,
    verbose=True,  # imprime cada passo aceito ao vivo, ver célula de markdown acima
)

print(f"\nVariáveis selecionadas: {resultado.variaveis}")
print(f"KS dev:   {resultado.ks_dev:.4f}")
print(f"KS teste: {resultado.ks_teste:.4f}")
print(f"AUC teste: {resultado.auc_teste:.4f}")
print(f"Taxa de evento dev/teste: {resultado.taxa_evento_dev:.2%} / {resultado.taxa_evento_teste:.2%}")

## 10. Resultados

In [ ]:
print("Histórico da busca (um evento por passo aceito, na ordem em que aconteceram):")
display(pd.DataFrame({"passo": range(1, len(resultado.historico) + 1), "evento": resultado.historico}))

print("Coeficientes:")
display(pd.Series(resultado.coeficientes, name="coeficiente").to_frame())

print("Estatísticas (coeficiente / erro padrão / p-valor — diagnóstico, não influencia a seleção):")
display(pd.DataFrame(resultado.estatisticas).T)

print("Tabela de decis (gains/KS table) na base de teste:")
display(pd.DataFrame(resultado.tabela_decis))

## Notas e próximos passos

- **Bucket `"val"`**: reservado de propósito, não usado neste notebook. Hoje `pipeline_lab.categorizar` ajusta e aplica a transformação WOE numa única chamada (`categorizar_e_transformar`) — não existe ainda uma função separada de "reaplicar as tabelas já ajustadas em dado novo, sem reajustar". Rodar `categorizar_e_transformar(val, val)` recalcularia o WOE usando o `y` do próprio `val` (vazamento). Se quiser avaliar nesse terceiro holdout ou gerar a submissão em cima de `churn_test` (sem rótulo), é preciso expor esse `aplicar_transformacoes(novos_dados, tabelas_do_dev)` na biblioteca antes — posso construir isso a seguir se for útil.
- **`test_pd`** (carregado no passo 1, vindo de `workspace.default.churn_test`) não foi usado neste notebook pelo mesmo motivo acima.
- Para ajustar qualquer hiperparâmetro, edite o valor no parâmetro nomeado correspondente em cada célula — todos já estão explícitos, não há nada escondido em default silencioso.